# Logistic Regression with Weka

Weka is the veteran — out of the University of Waikato since the late 1990s, and
the tool a whole generation learned machine learning on. Its age shows in the
API: where Smile took a `double[][]` and Tribuo took typed `Example`s, Weka wants
its own `Instances` structure, built from a list of `Attribute`s with the class
declared as a **nominal** attribute. It's more ceremony. It's also rock solid and
documented to death.

As always, the shared `.java` files do the data work. This notebook only does the
Weka-specific part: **adapt our feature table into `Instances`, train
`functions.Logistic`, and read the result.**

In [1]:
%%loadFromPOM
<dependency>
    <groupId>com.fasterxml.jackson.dataformat</groupId>
    <artifactId>jackson-dataformat-csv</artifactId>
    <version>2.17.2</version>
</dependency>
<dependency>
    <groupId>nz.ac.waikato.cms.weka</groupId>
    <artifactId>weka-stable</artifactId>
    <version>3.8.6</version>
</dependency>

## The shared pipeline

Identical to every other notebook in this post.

In [2]:
%load shared/Match.java
%load shared/DataLoader.java
%load shared/FormerName.java
%load shared/TeamNames.java
%load shared/EloRating.java
%load shared/RecentForm.java
%load shared/FeatureRow.java
%load shared/FeatureEngineering.java
%load shared/TrainTestSplit.java
%load shared/Metrics.java
%load shared/Predictions.java

## Weka imports

We import the exact Weka types we use, not `weka.core.*`. The wildcard form — and
even some explicit Weka imports — can make the notebook's Java engine (JShell)
re-resolve the namespace and drop earlier `var` bindings. To stay safe we put all
Weka imports in their own cell *before* we create any `var` we care about, so
nothing important exists yet to be dropped.

In [3]:
import weka.core.Attribute;
import weka.core.Instances;
import weka.core.DenseInstance;
import weka.core.Instance;
import weka.classifiers.functions.Logistic;
import java.util.ArrayList;
System.out.println("weka imports ready");

weka imports ready


## Build features and the Weka datasets

We run the shared pipeline and the Weka adapter together, in one cell, so the
`split`/`fe` variables are created and consumed without any import landing
between them.

Weka's data model is `Instances`: a named, schema-carrying table. We declare one
numeric `Attribute` per feature plus a final **nominal** class attribute with
values `{"0", "1"}`. Each `FeatureRow` becomes a `DenseInstance` whose last slot
holds the class index. A small local helper fills an `Instances` from a list of
rows; the upcoming fixtures get a *missing* class value, because we genuinely
don't know the result.

In [4]:
var all = DataLoader.loadAll("/home/jovyan/data/results.csv");
var names = TeamNames.load("/home/jovyan/data/former_names.csv");
var fe = FeatureEngineering.build(all, names);
var split = TrainTestSplit.chronological(fe.played(), 0.8);

String[] featureNames = FeatureRow.featureNames();

// Schema: numeric features + a nominal class {0,1}.
var attributes = new ArrayList<Attribute>();
for (String f : featureNames) attributes.add(new Attribute(f));
var classValues = new ArrayList<String>();
classValues.add("0");
classValues.add("1");
attributes.add(new Attribute("homeWin", classValues));
int classIndex = attributes.size() - 1;

var template = new Instances("football", attributes, 0);
template.setClassIndex(classIndex);

// Fill an Instances from rows. known=false -> class left missing (upcoming).
java.util.function.BiFunction<java.util.List<FeatureRow>, Boolean, Instances> fill =
    (rows, known) -> {
        Instances data = new Instances(template, rows.size());
        for (FeatureRow r : rows) {
            double[] vals = new double[classIndex + 1];
            double[] f = r.features();
            for (int j = 0; j < f.length; j++) vals[j] = f[j];
            Instance inst = new DenseInstance(1.0, vals);
            inst.setDataset(data);
            if (known) inst.setClassValue(r.homeWin() ? "1" : "0");
            else inst.setClassMissing();
            data.add(inst);
        }
        return data;
    };

Instances trainData = fill.apply(split.train(), true);
Instances testData = fill.apply(split.test(), true);
Instances upcomingData = fill.apply(fe.upcoming(), false);
int[] testY = TrainTestSplit.toY(split.test());
var upcomingFixtures = fe.upcoming();

System.out.println("train: " + trainData.numInstances() + "   test: " + testData.numInstances()
    + "   upcoming: " + upcomingData.numInstances());
System.out.println("class attribute: " + trainData.classAttribute());

train: 39546   test: 9887   upcoming: 44
class attribute: @attribute homeWin {0,1}


## Train

`weka.classifiers.functions.Logistic` is Weka's logistic regression. Training is
`buildClassifier(trainData)` — one line, once the `Instances` are shaped.

In [5]:
var model = new Logistic();
model.buildClassifier(trainData);

## Evaluate

`distributionForInstance` returns the class-probability array; the index of class
`"1"` is our P(home win). We feed those probabilities to the shared `Metrics` so
Weka is scored exactly like the others.

In [6]:
int homeWinIndex = trainData.classAttribute().indexOfValue("1");
double[] testProbs = new double[testData.numInstances()];
for (int i = 0; i < testData.numInstances(); i++) {
    double[] dist = model.distributionForInstance(testData.instance(i));
    testProbs[i] = dist[homeWinIndex];
}

var metrics = Metrics.from(testProbs, testY);
System.out.println("Weka logistic regression");
System.out.println(metrics);

Weka logistic regression
n=9887  accuracy=0.711  precision=0.680  recall=0.742  f1=0.710  logLoss=0.5561  brier=0.1889


## Read the coefficients

Weka, like Smile, exposes the fitted coefficients — and `model.toString()` prints
a readable per-feature table with odds ratios already computed. Worth seeing once,
but with the same warning from the Smile notebook: our features are unscaled and
collinear, so these coefficients are not safe to read as causal "insights." The
model predicts well; its story about *why* is not trustworthy. Accuracy and
interpretability are separate properties.

In [7]:
System.out.println(model.toString());

Logistic Regression with ridge parameter of 1.0E-8
Coefficients...
                  Class
Variable              0
eloDiff         -0.0059
homeWinRate      0.9452
awayWinRate     -0.8751
homeGoalDiff    -0.1832
awayGoalDiff     0.2325
neutral          0.3815
Intercept       -0.0551


Odds Ratios...
                  Class
Variable              0
eloDiff          0.9941
homeWinRate      2.5734
awayWinRate      0.4168
homeGoalDiff     0.8326
awayGoalDiff     1.2618
neutral          1.4644



## Predict the 2026 World Cup group stage

The 44 fixtures with no result — the 2026 World Cup group stage. Weka fills in
the probability for each.

In [8]:
double[] upcomingProbs = new double[upcomingData.numInstances()];
for (int i = 0; i < upcomingData.numInstances(); i++) {
    double[] dist = model.distributionForInstance(upcomingData.instance(i));
    upcomingProbs[i] = dist[homeWinIndex];
}
Predictions.print(upcomingFixtures, upcomingProbs);

date         home                   away                    P(home)   call
2026-06-19   Scotland               Morocco                   21.0%   no home win
2026-06-19   Brazil                 Haiti                     82.6%   Brazil win
2026-06-19   United States          Australia                 50.1%   United States win
2026-06-19   Turkey                 Paraguay                  48.9%   no home win
2026-06-20   Germany                Ivory Coast               64.1%   Germany win
2026-06-20   Ecuador                Curaçao                   88.5%   Ecuador win
2026-06-20   Netherlands            Sweden                    61.5%   Netherlands win
2026-06-20   Tunisia                Japan                     13.0%   no home win
2026-06-21   Belgium                Iran                      47.4%   no home win
2026-06-21   New Zealand            Egypt                     23.7%   no home win
2026-06-21   Spain                  Saudi Arabia              84.3%   Spain win
2026-06-21   Uru